In [1]:
from ta.volatility import AverageTrueRange

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from backtesting import Strategy
from backtesting import Backtest, set_bokeh_output
set_bokeh_output(notebook=False)

from datetime import timedelta

import sys
sys.path.append('../utils')
import Util
import importlib
importlib.reload(Util)

from enum import Enum
import time

from random import randint
import random


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
pair = 'GBPJPY'

df_d1_raw = pd.read_csv(f"../Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1_raw = pd.read_csv(f"../Trading Data/{pair}_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')

print(df_d1_raw)
print(df_h1_raw)


               Open     High      Low    Close  Tickvol
Date                                                   
2010-09-10  129.605  130.065  128.995  129.255   172229
2010-09-13  129.565  130.115  128.780  129.090   215859
2010-09-14  129.105  129.430  127.640  129.015   226170
2010-09-15  129.015  134.070  128.555  133.975   252317
2010-09-16  133.975  134.260  132.970  134.150   222443
...             ...      ...      ...      ...      ...
2026-09-18  208.323  211.276  208.233  210.095   364707
2026-09-21  210.296  210.762  209.676  210.321   243731
2026-09-22  210.321  210.892  209.513  210.080   333200
2026-09-23  210.080  210.176  209.422  209.564   273936
2026-09-24  209.566  210.105  208.768  209.848   199988

[4183 rows x 5 columns]
                        Open     High      Low    Close  Tickvol
Date                                                            
2018-08-07 11:00:00  144.154  144.204  144.088  144.134     5948
2018-08-07 12:00:00  144.133  144.149  144.012  144.

In [134]:
structure_d1_window = 5

date_format = '%Y-%m-%d %H:%M'
start_date = datetime.strptime('2019-1-1 00:00', date_format)
end_date = datetime.strptime('2025-12-30 00:00', date_format)

timezone = "Europe/London"

In [135]:
# process D1 data
df_d1 = df_d1_raw.loc[start_date - timedelta(days=30) : end_date]
indicator_atr_14 = AverageTrueRange(df_d1['High'], df_d1['Low'], df_d1['Close'], window=14)
df_d1['ATR'] = indicator_atr_14._atr

df_d1 = Util.implement_market_structure_states_fixed_pivots(df_d1, n=structure_d1_window)
df_d1["state_changed"] = df_d1["market_state"] != df_d1[
    "market_state"
].shift(1)
for state in df_d1[df_d1["state_changed"]][["Close", "market_state"]].iterrows():
    print(state)
df_d1 = df_d1.loc[start_date : end_date]

(Timestamp('2018-12-03 00:00:00'), Close           144.58
market_state       NaN
Name: 2018-12-03 00:00:00, dtype: object)
(Timestamp('2018-12-04 00:00:00'), Close           143.359
market_state        NaN
Name: 2018-12-04 00:00:00, dtype: object)
(Timestamp('2018-12-05 00:00:00'), Close           144.065
market_state        NaN
Name: 2018-12-05 00:00:00, dtype: object)
(Timestamp('2018-12-06 00:00:00'), Close           144.028
market_state        NaN
Name: 2018-12-06 00:00:00, dtype: object)
(Timestamp('2018-12-07 00:00:00'), Close           143.429
market_state        NaN
Name: 2018-12-07 00:00:00, dtype: object)
(Timestamp('2018-12-10 00:00:00'), Close           142.254
market_state        NaN
Name: 2018-12-10 00:00:00, dtype: object)
(Timestamp('2018-12-11 00:00:00'), Close           141.593
market_state        NaN
Name: 2018-12-11 00:00:00, dtype: object)
(Timestamp('2018-12-12 00:00:00'), Close           143.068
market_state        NaN
Name: 2018-12-12 00:00:00, dtype: object)
(T

In [136]:
df_h1 = df_h1_raw.loc[start_date : end_date]
index_list = []

for index, row in df_h1.iterrows():
    # ajust to daylight saving time
    dst = pd.Timestamp(index, tz=timezone).dst()
    if dst == timedelta(0):
        index = index - timedelta(hours=1)
        index_list.append(index)
    else:
        index_list.append(index)

df_h1.index = index_list

In [168]:
# process h1 data
horizon = 12
abn_list= []

for i in range(1, len(df_d1)):
    candle_daily = df_d1.iloc[i]
    previous_candle = df_d1.iloc[i-1]
    pdh = previous_candle['High']

    start_date_daily = candle_daily.name
    end_date_daily = start_date_daily + timedelta(hours=23, minutes=59)
    df_by_day = df_h1.loc[f'{start_date_daily}' : f'{end_date_daily}']

    breakout_candle_index = None
    # if previous_candle['market_state'] == "BEARISH":
    for j in range(0, len(df_by_day)):
        candle_hourly = df_by_day.iloc[j]

        if candle_hourly['High'] > pdh and candle_hourly['Open'] < pdh and breakout_candle_index is None:
            breakout_candle_index = j

        if (candle_hourly['High'] > pdh and 
            candle_hourly['Close'] < pdh and
            breakout_candle_index is not None and
            j - breakout_candle_index <= 1):
            entry_index = j + 1
            if entry_index > len(df_by_day) - 1:
                entry_index = len(df_by_day) - 1

            entry = df_by_day.iloc[entry_index]

            close_index = j + horizon
            if close_index > len(df_by_day) - 1:
                close_index = len(df_by_day) - 1
            close = (entry['Open'] - df_by_day['Close'].iloc[close_index]) / previous_candle['ATR']

            # print(f"{entry.name} {entry['Open']} {close}")
            abn_list.append(close)
            breakout_candle_index= None
            break

print(f"len = {len(abn_list)}")
df_result = pd.DataFrame({'abn' : abn_list})
print(f"positive = {(df_result['abn'] > 0).mean() * 100}")
print(f"max = {np.max(abn_list)}")
print(f"min = {np.min(abn_list)}")
print(f"mean = {np.mean(abn_list)}")

len = 570
positive = 50.35087719298246
max = 3.44203422990401
min = -2.393851114688487
mean = -0.023353149517634706


In [169]:
# process h1 data
horizon = 12
abn_list= []

for i in range(1, len(df_d1)):
    candle_daily = df_d1.iloc[i]
    previous_candle = df_d1.iloc[i-1]
    pdl = previous_candle['Low']

    start_date_daily = candle_daily.name
    end_date_daily = start_date_daily + timedelta(hours=23, minutes=59)
    df_by_day = df_h1.loc[f'{start_date_daily}' : f'{end_date_daily}']

    breakout_candle_index = None
    # if previous_candle['market_state'] == "BULLISH":
    for j in range(0, len(df_by_day)):
        candle_hourly = df_by_day.iloc[j]

        if candle_hourly['Low'] < pdl and candle_hourly['Open'] > pdl and breakout_candle_index is None:
            breakout_candle_index = j

        if (candle_hourly['Low'] < pdl and 
            candle_hourly['Close'] > pdl and
            breakout_candle_index is not None and
            j - breakout_candle_index <= 1):
            entry_index = j + 1
            if entry_index > len(df_by_day) - 1:
                entry_index = len(df_by_day) - 1

            entry = df_by_day.iloc[entry_index]

            close_index = j + horizon
            if close_index > len(df_by_day) - 1:
                close_index = len(df_by_day) - 1
            close = (df_by_day['Close'].iloc[close_index]) - entry['Open'] / previous_candle['ATR']

            # print(f"{entry.name} {entry['Open']} {close}")
            abn_list.append(close)
            breakout_candle_index= None
            break

print(f"len = {len(abn_list)}")
df_result = pd.DataFrame({'abn' : abn_list})
print(f"positive = {(df_result['abn'] > 0).mean() * 100}")
print(f"max = {np.max(abn_list)}")
print(f"min = {np.min(abn_list)}")
print(f"mean = {np.mean(abn_list)}")

len = 469
positive = 93.81663113006397
max = 124.60620315350835
min = -32.26317311950797
mean = 48.3240024037001


In [157]:
entry_hour_list = []
profit_list = []
horizon = 12

for i in range(1, len(df_d1)):
    candle_daily = df_d1.iloc[i]
    previous_candle = df_d1.iloc[i-1]

    start_date_daily = candle_daily.name
    end_date_daily = start_date_daily + timedelta(hours=23, minutes=59)
    df_by_day = df_h1.loc[f'{start_date_daily}' : f'{end_date_daily}']

    breakout_candle_index = None
    for j in range(0, len(df_by_day)-1):
        candle_hourly = df_by_day.iloc[j]
        entry_index = j + 1

        close_index = j + horizon
        if close_index > len(df_by_day) - 1:
            close_index = len(df_by_day) - 1

        profit = (df_by_day['Open'].iloc[entry_index] - df_by_day['Close'].iloc[close_index]) / previous_candle['ATR']
        entry_hour_list.append(candle_hourly.name.hour)
        profit_list.append(profit)

df_result = pd.DataFrame({'hour' : entry_hour_list, 'profit' : profit_list})
# print(df_result)
print(df_result.groupby('hour')['profit'].mean())

hour
0     0.003189
1    -0.003945
2    -0.012802
3    -0.003708
4    -0.016571
5    -0.022138
6    -0.021286
7    -0.018057
8    -0.028863
9    -0.018190
10    0.002877
11   -0.008536
12   -0.010369
13   -0.009666
14   -0.004849
15   -0.006592
16    0.000943
17    0.005464
18    0.007414
19    0.006185
20    0.013330
21   -0.017242
22   -0.019593
Name: profit, dtype: float64


In [158]:
entry_hour_list = []
profit_list = []
horizon = 12

for i in range(1, len(df_d1)):
    candle_daily = df_d1.iloc[i]
    previous_candle = df_d1.iloc[i-1]

    start_date_daily = candle_daily.name
    end_date_daily = start_date_daily + timedelta(hours=23, minutes=59)
    df_by_day = df_h1.loc[f'{start_date_daily}' : f'{end_date_daily}']

    breakout_candle_index = None
    for j in range(0, len(df_by_day)-1):
        candle_hourly = df_by_day.iloc[j]
        entry_index = j + 1

        close_index = j + horizon
        if close_index > len(df_by_day) - 1:
            close_index = len(df_by_day) - 1

        profit = (df_by_day['Close'].iloc[close_index] - df_by_day['Open'].iloc[entry_index]) / previous_candle['ATR']
        entry_hour_list.append(candle_hourly.name.hour)
        profit_list.append(profit)

df_result = pd.DataFrame({'hour' : entry_hour_list, 'profit' : profit_list})
# print(df_result)
print(df_result.groupby('hour')['profit'].mean())

hour
0    -0.003189
1     0.003945
2     0.012802
3     0.003708
4     0.016571
5     0.022138
6     0.021286
7     0.018057
8     0.028863
9     0.018190
10   -0.002877
11    0.008536
12    0.010369
13    0.009666
14    0.004849
15    0.006592
16   -0.000943
17   -0.005464
18   -0.007414
19   -0.006185
20   -0.013330
21    0.017242
22    0.019593
Name: profit, dtype: float64
